# Baseline evaluation 

Анализ прогона моделей на данных -- результаты прогона лежат в файлах 
-  /Users/arkuchina/diplom_data/LLM_Stats/results/test_or_deepseek.jsonl
-  /Users/arkuchina/diplom_data/LLM_Stats/results/sanity_gemini.jsonl

В данном ноутбуке производится замер метрик для каждого этапа пайплайна, а также анализ ошибок и галлюцинаций, для того чтобы 
1) улучшить бейзлайн быстрыми правкам и сенити-чеками 
2) сформировать список эксперментов, которые улучшат метрики 



## Выводы и дальнейший план действия 

1. DeepSeek: F1 = 0.84, но Hallucination = 0.16. Стиль галлюцинаций — 42% ложных предсказаний это дубликаты (один и тот же тест извлечён несколько раз, обычно из одной таблицы).            
2. Gemini: высокий Recall = 0.94, но Hallucination = 0.26. Стиль другой — 55% ложных это «правильный тип теста, но неправильное число» (off_stat_value), модель over-extract'ит и подкручивает значения.             
3. На реальных статьях DeepSeek проваливается (F1 = 0.19) — не справляется с длинными SPSS-таблицами.

Вывод: быстрый фикс в бейзлайне, который поможет улучшить метрики 
1. Дедупликация — если два предсказания совпадают по типу теста и значению статистики (с округлением до сотых), оставляем только первое. Это убирает повторные извлечения из таблиц.                                 
2. Фильтр неправдоподобных значений — отсеиваем предсказания с физически невозможными числами: p вне [0, 1],отрицательные df и тп -  Это убирает off_stat_value галлюцинации.   

Запланированные эксперименты:
1. Суммаризация
2. Chain-of-Thougnt на 4 этапе 
3. Кросс-проверка моделей 

## 1. Загрузка и индексация


In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd

In [ ]:
REPO = Path('.').resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
print(REPO)

# какие файлы грузим
RESULTS_FILES = [
    REPO / 'results/test_or_deepseek.jsonl',
    REPO / 'results' / 'sanity_gemini.jsonl',
]
RESULTS_FILES = [p for p in RESULTS_FILES if p.exists()]
print('Файлов найдено:', len(RESULTS_FILES))
for p in RESULTS_FILES:
    print(' ', p)

/Users/arkuchina/diplom_data/LLM_Stats
Файлов найдено: 2
  /Users/arkuchina/diplom_data/LLM_Stats/results/test_or_deepseek.jsonl
  /Users/arkuchina/diplom_data/LLM_Stats/results/sanity_gemini.jsonl


In [ ]:
rows = []
for p in RESULTS_FILES:
    with p.open() as f:
        for line in f:
            r = json.loads(line)
            r['_file'] = p.name
            rows.append(r)

df = pd.DataFrame(rows)
print('Всего записей:', len(df))
print(df.groupby(['model', 'source']).size().rename('n').to_frame())

Всего записей: 504
                                 n
model               source        
openrouter-deepseek real         2
                    synthetic  250
openrouter-gemini   real         2
                    synthetic  250


In [ ]:
errors = df[df['error'].notna()]
ok = df[df['error'].isna()].reset_index(drop=True)
print('OK:', len(ok), ', ошибок пайплайна:', len(errors))
if len(errors):
    display(errors[['example_id', 'source', 'error']].head(10))

OK: 503, ошибок пайплайна: 1


,example_id,source,error
114,test-syn-114,synthetic,JSONDecodeError: Extra data: line 3 column 1 (...


## 2. Общая сводка метрик


In [ ]:
from pipeline import eval as eval_mod


def aggregate_df(group):
    s3 = eval_mod.aggregate_stage3([r for r in group['stage3'] if r])
    s4 = eval_mod.aggregate_stage4([r for r in group['stage4'] if r])
    s5 = eval_mod.aggregate_stage5([r for r in group['stage5'] if r])
    s1 = [r for r in group['stage1'] if r]
    cov = None
    if s1:
        m = sum(x['matched'] for x in s1)
        t = sum(x['total'] for x in s1)
        if t:
            cov = m / t
    n_gold = sum(group['n_gold_tests'])
    n_pred = sum(group['n_predicted_tests'])
    return pd.Series({
        'n_examples': len(group),
        'n_gold_tests': n_gold,
        'n_predicted_tests': n_pred,
        'coverage_s1': cov,
        'precision': s3.get('precision'),
        'recall': s3.get('recall'),
        'f1': s3.get('f1'),
        'field_acc': s3.get('field_accuracy'),
        'complete_extr': s3.get('complete_extraction_rate'),
        'hallucination': s3.get('hallucination_rate'),
        'dir_acc': s4.get('primary_direction_accuracy'),
        'found_interp': s4.get('found_interpretation_rate'),
        'IDR': s5.get('inconsistency_detection_rate'),
        'FAR': s5.get('false_alarm_rate'),
    })


summary = ok.groupby(['model', 'source']).apply(aggregate_df).round(3)
summary

=== Сводка (model × source) ===


/var/folders/jr/rj82p0013hz9969tf03c80780000gn/T/ipykernel_27549/1440371247.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary = ok.groupby(['model', 'source']).apply(aggregate_df).round(3)


n_examples  n_gold_tests  n_predicted_tests  \
model               source                                                   
openrouter-deepseek real              2.0          58.0               17.0   
                    synthetic       249.0         268.0              308.0   
openrouter-gemini   real              2.0          53.0              100.0   
                    synthetic       250.0         286.0              335.0   

                               coverage_s1  precision  recall     f1  \
model               source                                             
openrouter-deepseek real               1.0      0.412   0.121  0.187   
                    synthetic          NaN      0.864   0.993  0.924   
openrouter-gemini   real               0.0      0.390   0.736  0.510   
                    synthetic          NaN      0.839   0.983  0.905   

                               field_acc  complete_extr  hallucination  \
model               source                                               
openrouter-deepseek real           0.571          0.000          0.588   
                    synthetic      0.948          0.741          0.136   
openrouter-gemini   real           0.738          0.256          0.610   
                    synthetic      0.954          0.769          0.161   

                               dir_acc  found_interp    IDR    FAR  
model               source                                          
openrouter-deepseek real         0.571          1.00    NaN    NaN  
                    synthetic      NaN          1.00  0.770  0.047  
openrouter-gemini   real         0.564          0.59    NaN  1.000  
                    synthetic      NaN          1.00  0.745  0.084

In [ ]:
summary_by_model = ok.groupby('model').apply(aggregate_df).round(3)
summary_by_model

=== Сводка по модели (все данные — synth + real) ===


/var/folders/jr/rj82p0013hz9969tf03c80780000gn/T/ipykernel_20936/1149508080.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  summary_by_model = ok.groupby('model').apply(aggregate_df).round(3)


,n_examples,n_gold_tests,n_predicted_tests,coverage_s1,precision,recall,f1,field_acc,complete_extr,hallucination,dir_acc,found_interp,IDR,FAR
model,,,,,,,,,,,,,,
openrouter-deepseek,251.0,326.0,325.0,1.0,0.840,0.837,0.839,0.938,0.722,0.160,0.571,1.00,0.770,0.047
openrouter-gemini,252.0,339.0,435.0,0.0,0.736,0.944,0.827,0.928,0.706,0.264,0.564,0.95,0.745,0.090


In [ ]:
df = summary_by_model
df.to_csv("summary_by_model.csv", index=False)
df.to_excel("summary_by_model.xlsx", index=False)

## 3. raw_text coverage по реальным статьям


In [ ]:
real = ok[ok['source'] == 'real'].copy()
if not len(real):
    print('Нет реальных записей в results.')
else:
    real['coverage'] = real['stage1'].apply(lambda x: x['coverage'] if x else None)
    real['matched'] = real['stage1'].apply(lambda x: x['matched'] if x else 0)
    real['total'] = real['stage1'].apply(lambda x: x['total'] if x else 0)
    real['markdown_len'] = real['markdown_stats'].apply(
        lambda x: x.get('markdown_len') if x else None)
    real['n_pages'] = real['markdown_stats'].apply(
        lambda x: x.get('n_pages') if x else None)
    real['n_tables'] = real['markdown_stats'].apply(
        lambda x: x.get('n_tables') if x else None)
    display(real[['model', 'source_id', 'n_pages', 'n_tables',
                  'markdown_len', 'matched', 'total', 'coverage']])

,model,source_id,n_pages,n_tables,markdown_len,matched,total,coverage
249,openrouter-deepseek,P3,27,7,102800,39,39,1.0
250,openrouter-deepseek,P2,13,7,46665,19,19,1.0
501,openrouter-gemini,P4,16,0,87646,0,12,0.0
502,openrouter-gemini,P1,12,7,66150,0,41,0.0


In [ ]:
# missed raw_text
for _, row in real.iterrows():
    if row['stage1']:
        missed = row['stage1'].get('missed', [])
    else:
        missed = []
    if missed:
        print('\n===', row['model'], 'x', row['source_id'],
              ': пропущено', len(missed), 'raw_text ===')
        for i, m in enumerate(missed[:5], 1):
            print('  [' + str(i) + ']', m[:150])
        if len(missed) > 5:
            print('  ... и ещё', len(missed) - 5)


=== openrouter-gemini × P4: пропущено 12 raw_text ===
  [1] Perceived Stress, Time: F(2, 112) = 35.75, p < .0001, partial η² = 0.50, 95% CI [0.36, 0.60]
  [2] Perceived Stress, Group: F(1, 56) = 28.42, p < .0001, partial η² = 0.48, 95% CI [0.31, 0.58]
  [3] Perceived Stress, Time × Group: F(2, 112) = 59.96, p < .0001, partial η² = 0.63, 95% CI [0.51, 0.70]
  [4] Mental Health (PCS), Time: F(2, 112) = 20.38, p < .0001, partial η² = 0.37, 95% CI [0.22, 0.49]
  [5] Mental Health (PCS), Group: F(1, 56) = 16.47, p < .0001, partial η² = 0.32, 95% CI [0.16, 0.45]
  ... и ещё 7

=== openrouter-gemini × P1: пропущено 41 raw_text ===
  [1] Work pace: F=11.816, Sig.=0.001 (COVID care 59.08 vs Not 49.78)
  [2] Influence at work: F=25.855, Sig.<0.001 (COVID care 38.18 vs Not 51.79)
  [3] Predictability: F=15.867, Sig.<0.001 (COVID care 44.71 vs Not 57.03)
  [4] Reward: F=8.710, Sig.=0.003 (COVID care 55.82 vs Not 65.03)
  [5] Role clarity: F=4.243, Sig.=0.040 (COVID care 70.19 vs Not 75.37)
  ... 

## 4. Анализ этапа 3 — извлечение тестов

###  Per-example метрики


In [ ]:
rows_s3 = []
for _, r in ok.iterrows():
    rec = {
        'example_id': r['example_id'],
        'model': r['model'],
        'source': r['source'],
        'source_id': r.get('source_id'),
    }
    rec.update(r['stage3'] or {})
    rows_s3.append(rec)

stage3_df = pd.DataFrame(rows_s3)
print('Колонки:', list(stage3_df.columns))
display(stage3_df.head())

Колонки: ['example_id', 'model', 'source', 'source_id', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'field_accuracy', 'complete_extraction_rate', 'hallucination_rate', 'pairs', 'fp_classifications']


,example_id,model,source,source_id,tp,fp,fn,precision,recall,f1,field_accuracy,complete_extraction_rate,hallucination_rate,pairs,fp_classifications
0,test-syn-000,openrouter-deepseek,synthetic,451,1,1,0,0.5,1.0,0.666667,1.0,1.0,0.5,"[[0, 0]]","[{'category': 'duplicate', 'pred': {'test_type..."
1,test-syn-001,openrouter-deepseek,synthetic,81,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
2,test-syn-002,openrouter-deepseek,synthetic,395,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
3,test-syn-003,openrouter-deepseek,synthetic,320,1,1,0,0.5,1.0,0.666667,1.0,1.0,0.5,"[[0, 0]]","[{'category': 'duplicate', 'pred': {'test_type..."
4,test-syn-004,openrouter-deepseek,synthetic,453,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,[],[]


### 4.2. Метрики в разрезе по `test_type` и по `environment`


In [ ]:
from pipeline.test_verificator import normalize_test_type

# Загружаем датасет, чтобы матчить обратно
DATASETS = {
    'dev': REPO / 'data' / 'dev_dataset.jsonl',
    'test': REPO / 'data' / 'test_dataset.jsonl',
}
gold_by_id = {}
for split_name, p in DATASETS.items():
    if not p.exists():
        continue
    with p.open() as f:
        for line in f:
            rec = json.loads(line)
            gold_by_id[rec['example_id']] = rec
print('Gold loaded:', len(gold_by_id), 'records')

Gold loaded: 504 records


In [ ]:
# per-test разбор
def match_one(pred, gold, rel_tol=0.02):
    if normalize_test_type(pred.get('test_type')) != normalize_test_type(gold.get('test_type')):
        return False
    a = pred.get('statistic_value')
    b = gold.get('statistic_value')
    if a is None or b is None:
        return False
    return abs(a - b) / max(abs(b), 1e-9) < rel_tol


per_test = []
per_pred = []
for _, row in ok.iterrows():
    ex_id = row['example_id']
    gold_rec = gold_by_id.get(ex_id)
    if gold_rec is None:
        continue
    gold_tests = gold_rec['tests']
    pred_tests_count = row['n_predicted_tests']
    gold_taken = [False] * len(gold_tests)
    tp = row['stage3'].get('tp', 0)
    fp = row['stage3'].get('fp', 0)
    fn = row['stage3'].get('fn', 0)
    for g in gold_tests:
        per_test.append({
            'example_id': ex_id,
            'source': row['source'],
            'environment': gold_rec.get('environment'),
            'error_type_gold': gold_rec.get('error_type'),
            'test_type': normalize_test_type(g.get('test_type')),
            'has_df1': g.get('df1') is not None,
            'has_df2': g.get('df2') is not None,
        })

per_test_df = pd.DataFrame(per_test)
per_test_df.head()

Gold тестов в разборе: 665


,example_id,source,environment,error_type_gold,test_type,has_df1,has_df2
0,test-syn-000,synthetic,text,none,chi,True,False
1,test-syn-003,synthetic,table,rounding,t,True,False
2,test-syn-005,synthetic,apa,wrong_pvalue,chi,True,False
3,test-syn-008,synthetic,two_text,none,r,True,False
4,test-syn-008,synthetic,two_text,none,t,True,False


In [ ]:
# recall по test_type и environment
def recall_by(group_col):
    rows = []
    for _, g in stage3_df.groupby(['example_id', 'model']):
        ex_id = g.iloc[0]['example_id']
        model = g.iloc[0]['model']
        rec = gold_by_id.get(ex_id)
        if rec is None or not rec['tests']:
            continue
        types = [normalize_test_type(t.get('test_type')) for t in rec['tests']]
        major_type = Counter(types).most_common(1)[0][0]
        out = {
            'example_id': ex_id,
            'model': model,
            'major_test_type': major_type,
            'environment': rec.get('environment'),
            'source': rec.get('source'),
        }
        out.update(g.iloc[0][['tp', 'fp', 'fn']].to_dict())
        rows.append(out)
    df = pd.DataFrame(rows)
    return df.groupby(['model', group_col]).agg(
        n=('example_id', 'count'),
        tp=('tp', 'sum'),
        fp=('fp', 'sum'),
        fn=('fn', 'sum'),
    ).assign(
        recall=lambda x: x['tp'] / (x['tp'] + x['fn']).replace(0, pd.NA),
        precision=lambda x: x['tp'] / (x['tp'] + x['fp']).replace(0, pd.NA),
        f1=lambda x: 2 * x['precision'] * x['recall'] / (x['precision'] + x['recall']).replace(0, pd.NA),
    ).round(3)


display(recall_by('major_test_type'))
print()
display(recall_by('environment'))

Разрез по test_type:


n  tp  fp  fn  recall  precision     f1
model               major_test_type                                          
openrouter-deepseek F                23  39  10  35   0.527      0.796  0.634
                    Q                32  51   9   0   1.000      0.850  0.919
                    chi              23  37   5   0   1.000      0.881  0.937
                    r                26  47  10   2   0.959      0.825  0.887
                    t                30  52  12  16   0.765      0.812  0.788
                    z                28  47   6   0   1.000      0.887  0.940
openrouter-gemini   F                24  79  72  14   0.849      0.523  0.648
                    Q                36  64  15   0   1.000      0.810  0.895
                    chi              24  39   4   3   0.929      0.907  0.918
                    r                36  63  13   1   0.984      0.829  0.900
                    t                22  43   5   0   1.000      0.896  0.945
                    z                20  32   6   1   0.970      0.842  0.901


Разрез по environment:


n  tp  fp  fn  recall  precision     f1
model               environment                                          
openrouter-deepseek apa          39  58   6   0   1.000      0.906  0.951
                    non_apa      27  41   3   0   1.000      0.932  0.965
                    table        21  32  20   0   1.000      0.615  0.762
                    text         37  65   8   0   1.000      0.890  0.942
                    two_apa      21  42   2   0   1.000      0.955  0.977
                    two_text     15  28   3   2   0.933      0.903  0.918
openrouter-gemini   apa          41  70   6   0   1.000      0.921  0.959
                    non_apa      25  42   1   1   0.977      0.977  0.977
                    table        20  37  33   0   1.000      0.529  0.692
                    text         37  58   8   4   0.935      0.879  0.906
                    two_apa      22  44   5   0   1.000      0.898  0.946
                    two_text     15  30   1   0   1.000      0.968  0.984

### 4.3. Виды галлюцинаций (FP)

Каждый FP классифицирован  по 4 категориям:

- **`duplicate`** — LLM вернул тест, который уже matched другой записью → дублирование
- **`wrong_test_type`** — statistic_value совпадает с каким-то  из датасета, но тип теста перепутан (напр. t вместо F)
- **`off_stat_value`** — тип теста правильный, но число неверное (опечатка / округление / подмена)
- **`complete_fabrication`** — ни по типу, ни по значению не близок ни к одному gold → полностью выдуманный тест


In [ ]:
# сводка по видам галлюцинаций
fp_records = []
for _, row in ok.iterrows():
    fps = (row['stage3'] or {}).get('fp_classifications', [])
    for fp in fps:
        pred = fp.get('pred') or {}
        closest = fp.get('closest_gold') or {}
        fp_records.append({
            'example_id': row['example_id'],
            'source': row['source'],
            'model': row['model'],
            'category': fp['category'],
            'pred_test_type': pred.get('test_type'),
            'pred_stat': pred.get('statistic_value'),
            'pred_df1': pred.get('df1'),
            'pred_df2': pred.get('df2'),
            'pred_p': pred.get('reported_p'),
            'pred_raw_text': (pred.get('raw_text') or '')[:150],
            'closest_gold_type': closest.get('test_type'),
            'closest_gold_stat': closest.get('statistic_value'),
        })

fp_df = pd.DataFrame(fp_records)

if len(fp_df):
    display(fp_df.groupby(['model', 'category']).size().rename('count').to_frame())

    print('\nДоля категории от всех FP (в процентах):')
    pct = (fp_df.groupby(['model', 'category']).size() /
           fp_df.groupby('model').size() * 100).round(1)
    display(pct.rename('pct').to_frame())

Всего FP-записей: 167

Сводка по видам галлюцинаций (model × category):


count
model               category                   
openrouter-deepseek complete_fabrication     18
                    duplicate                22
                    off_stat_value           12
openrouter-gemini   complete_fabrication     17
                    duplicate                35
                    off_stat_value           63


Доля категории от всех FP (в процентах):


pct
model               category                  
openrouter-deepseek complete_fabrication  34.6
                    duplicate             42.3
                    off_stat_value        23.1
openrouter-gemini   complete_fabrication  14.8
                    duplicate             30.4
                    off_stat_value        54.8

In [ ]:
if len(fp_df):
    wrong_type = fp_df[fp_df['category'] == 'wrong_test_type']
    if len(wrong_type):
        confusion = wrong_type.groupby(['closest_gold_type', 'pred_test_type']).size()
        confusion = confusion.rename('n').to_frame().sort_values('n', ascending=False)
        print('Confusion matrix для wrong_test_type (gold -> pred):')
        display(confusion)
    else:
        print('Нет wrong_test_type - LLM не путает типы тестов.')

Нет wrong_test_type — LLM не путает типы тестов.


In [ ]:
# примеры по 3 на category
if len(fp_df):
    for cat in ['complete_fabrication', 'off_stat_value', 'wrong_test_type', 'duplicate']:
        sample = fp_df[fp_df['category'] == cat].head(3)
        if not len(sample):
            continue
        n_total = len(fp_df[fp_df.category == cat])
        print('\n===', cat, '(', n_total, 'всего ) ===')
        for _, row in sample.iterrows():
            print('  [' + str(row['example_id']) + ']',
                  'pred:', str(row['pred_test_type']) + '(' +
                  str(row['pred_df1']) + ',' + str(row['pred_df2']) + ')=' +
                  str(row['pred_stat']) + ', p=' + str(row['pred_p']))
            if row['closest_gold_type'] is not None:
                print('    closest gold:', str(row['closest_gold_type']) +
                      '=' + str(row['closest_gold_stat']))
            if row['pred_raw_text']:
                print('    raw_text:', row['pred_raw_text'])


=== complete_fabrication (35 всего) ===
  [test-syn-012] pred: t(nan,nan)=2.81, p=0.006
    raw_text: t = 2.81, p = 0.006
  [test-syn-021] pred: Q(nan,nan)=2.18, p=nan
    raw_text: heterogeneity tests revealed no meaningful moderation effects (Q = 2.18, p > .05).
  [test-syn-044] pred: r(nan,nan)=0.68, p=nan
    raw_text: financial services firms exhibited the strongest positive correlation between dividend payouts and ROE (r = 0.68)

=== off_stat_value (75 всего) ===
  [test-syn-137] pred: t(244.0,nan)=0.34, p=0.73
    closest gold: t=4.58
    raw_text: t(244) = 0.34, p = .73
  [test-syn-157] pred: r(nan,nan)=-0.53, p=nan
    closest gold: r=-0.49
    raw_text: r = -0.53
  [test-syn-157] pred: r(nan,nan)=-0.45, p=nan
    closest gold: r=-0.49
    raw_text: r = -0.45

=== duplicate (57 всего) ===
  [test-syn-000] pred: chi(nan,nan)=2.91, p=0.99
    closest gold: Chi2=2.91
    raw_text: the test statistic remained near 2.91 with p-values exceeding 0.99
  [test-syn-003] pred: t(17.0,na

In [ ]:
# на каких environment чаще галлюцинирует
if len(fp_df):
    fp_with_env = fp_df.copy()
    fp_with_env['environment'] = fp_with_env['example_id'].map(
        lambda ex: (gold_by_id.get(ex) or {}).get('environment')
    )
    per_env = fp_with_env.groupby(['environment', 'category']).size().unstack(fill_value=0)
    print('FP по environment x category:')
    display(per_env)

FP по environment × category:


category,complete_fabrication,duplicate,off_stat_value
environment,,,
apa,9,0,3
non_apa,1,0,3
table,3,50,0
text,12,1,3
two_apa,6,0,1
two_text,4,0,0


## 5. Этап 4 — интерпретации


In [ ]:
# confusion primary_direction (только real)
rows_s4 = []
for _, row in ok.iterrows():
    if row['source'] != 'real':
        continue
    gold_rec = gold_by_id.get(row['example_id'])
    if not gold_rec:
        continue
    s4 = row.get('stage4') or {}
    rows_s4.append({
        'example_id': row['example_id'],
        'source_id': row['source_id'],
        'matched_pairs': s4.get('matched_pairs'),
        'direction_hits': s4.get('direction_hits'),
        'direction_total': s4.get('direction_total'),
        'accuracy': s4.get('primary_direction_accuracy'),
        'found_interp_rate': s4.get('found_interpretation_rate'),
    })

pd.DataFrame(rows_s4).round(3)

,example_id,source_id,matched_pairs,direction_hits,direction_total,accuracy,found_interp_rate
0,real-multicomponent,P3,4,1,4,0.250,1.000
1,real-sleep_deprivation,P2,3,3,3,1.000,1.000
2,real-mbct,P4,6,6,6,1.000,1.000
3,real-stress_burnout,P1,33,16,33,0.485,0.515


## 6. Этап 5 — проверка согласованности результатов и словесного описания


In [ ]:
rows_s5 = []
for _, row in ok.iterrows():
    gold_rec = gold_by_id.get(row['example_id'])
    if gold_rec is None:
        continue
    s5 = row.get('stage5') or {}
    rows_s5.append({
        'example_id': row['example_id'],
        'model': row['model'],
        'source': row['source'],
        'environment': gold_rec.get('environment'),
        'error_type_gold': gold_rec.get('error_type'),
        'tpv': s5.get('tpv', 0),
        'fpv': s5.get('fpv', 0),
        'fnv': s5.get('fnv', 0),
        'not_checkable': s5.get('not_checkable', 0),
    })

s5_df = pd.DataFrame(rows_s5)

# IDR по (model x error_type)
print('IDR / FAR по (модель x тип внесённой ошибки):')
agg = s5_df.groupby(['model', 'error_type_gold']).agg(
    n=('example_id', 'count'),
    tpv=('tpv', 'sum'),
    fpv=('fpv', 'sum'),
    fnv=('fnv', 'sum'),
    nc=('not_checkable', 'sum'),
).assign(
    IDR=lambda x: x['tpv'] / (x['tpv'] + x['fnv']).replace(0, pd.NA),
    FAR=lambda x: x['fpv'] / (x['fpv'] + x['tpv']).replace(0, pd.NA),
).round(3)
display(agg)

print()
print('FAR по (модель x environment), только примеры БЕЗ внесённых ошибок:')
no_err = s5_df[s5_df['error_type_gold'].isin(['none', None])]
agg2 = no_err.groupby(['model', 'environment']).agg(
    n=('example_id', 'count'),
    fpv=('fpv', 'sum'),
).round(3)
display(agg2)

IDR / FAR по (модель × тип внесённой ошибки):


n  tpv  fpv  fnv  nc    IDR    FAR
model               error_type_gold                                       
openrouter-deepseek none              171   85    3   21   7  0.802  0.034
                    rounding           19   18    2    6   3  0.750  0.100
                    transcription      17   14    0    8   1  0.636  0.000
                    wrong_conclusion   18   25    1    4   0  0.862  0.038
                    wrong_pvalue       24   19    2    9   2  0.679  0.095
openrouter-gemini   none              172   87    4   26   5  0.770  0.044
                    rounding           20   20    1    6   0  0.769  0.048
                    transcription      15    9    4    7   1  0.562  0.308
                    wrong_conclusion   20   21    1    7   1  0.750  0.045
                    wrong_pvalue       23   15    4    6   1  0.714  0.211


FAR по (модель × environment), только примеры БЕЗ внесённых ошибок:


n  fpv
model               environment         
openrouter-deepseek apa          17    0
                    no_test      89    0
                    non_apa      15    0
                    table        11    1
                    text         19    1
                    two_apa      14    1
                    two_text      6    0
openrouter-gemini   apa          17    1
                    no_test      90    0
                    non_apa      14    0
                    table        12    0
                    text         19    0
                    two_apa      14    3
                    two_text      6    0

## 7. Анализ конкретных ошибок 


In [ ]:
stage3_df['errors_s3'] = stage3_df['fp'].fillna(0) + stage3_df['fn'].fillna(0)
worst_s3 = stage3_df.sort_values('errors_s3', ascending=False).head(10)
display(worst_s3[['example_id', 'source', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1']])

,example_id,source,tp,fp,fn,precision,recall,f1
502,real-stress_burnout,real,33,55,8,0.375000,0.804878,0.511628
249,real-multicomponent,real,4,7,35,0.363636,0.102564,0.160000
250,real-sleep_deprivation,real,3,3,16,0.500000,0.157895,0.240000
501,real-mbct,real,6,6,6,0.500000,0.500000,0.500000
460,dev-syn-209,synthetic,3,3,0,0.500000,1.000000,0.666667
304,dev-syn-053,synthetic,3,3,0,0.500000,1.000000,0.666667
254,dev-syn-003,synthetic,0,0,3,NaN,0.000000,NaN
44,test-syn-044,synthetic,1,3,0,0.250000,1.000000,0.400000
237,test-syn-238,synthetic,3,3,0,0.500000,1.000000,0.666667
66,test-syn-066,synthetic,3,3,0,0.500000,1.000000,0.666667


In [ ]:
if len(worst_s3):
    EX_ID = worst_s3.iloc[0]['example_id']
else:
    EX_ID = None
print(EX_ID)

if EX_ID:
    result_row = ok[ok['example_id'] == EX_ID].iloc[0]
    gold_rec = gold_by_id.get(EX_ID, {})

    print('\n-- FRAGMENT / text --')
    print((gold_rec.get('fragment', '') or '')[:1500])

    print('\n-- GOLD tests --')
    for i, t in enumerate(gold_rec.get('tests', [])):
        print('  [' + str(i) + ']',
              str(t.get('test_type')) + '(' + str(t.get('df1')) + ',' +
              str(t.get('df2')) + ')=' + str(t.get('statistic_value')) + ',',
              'p=' + str(t.get('p_equality', '=')) + str(t.get('reported_p')),
              ' consistent=' + str(t.get('consistent')),
              ' err=' + str(t.get('error_type')))

    print('\n-- STAGE 3 metrics --')
    print(result_row['stage3'])

    print('\n-- STAGE 5 flags (verified) --')
    print(result_row['stage5'])

Разбираем: real-stress_burnout

-- FRAGMENT / text --
Work pace: F=11.816, Sig.=0.001 (COVID care 59.08 vs Not 49.78)

Influence at work: F=25.855, Sig.<0.001 (COVID care 38.18 vs Not 51.79)

Predictability: F=15.867, Sig.<0.001 (COVID care 44.71 vs Not 57.03)

Reward: F=8.710, Sig.=0.003 (COVID care 55.82 vs Not 65.03)

Role clarity: F=4.243, Sig.=0.040 (COVID care 70.19 vs Not 75.37)

Role conflicts: F=15.268, Sig.<0.001 (COVID care 55.21 vs Not 45.93)

Quality of leadership: F=6.944, Sig.=0.009 (COVID care 57.49 vs Not 65.57)

Social support from supervisor: F=4.201, Sig.=0.041 (COVID care 59.24 vs Not 65.55)

Job satisfaction: F=11.709, Sig.=0.001 (COVID care 54.36 vs Not 62.84)

Trust regarding management: F=20.497, Sig.<0.001 (COVID care 55.89 vs Not 67.86)

Justice and respect: F=11.831, Sig.=0.001 (COVID care 44.51 vs Not 54.35)

Stress: F=4.263, Sig.=0.040 (COVID care 47.96 vs Not 42.35)

Burnout: F=1.923, Sig.=0.167

Sleeping troubles: F=0.855, Sig.=0.356

Profession x COVID 